In [1]:
# === Cellule 1 : Librairies essentielles et chargement du CSV ===
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Charger le fichier CSV
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv", sep=';')
print("Aperçu des données :")
display(df.head())


Aperçu des données :


,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.6
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.5
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.6
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.6
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.1


In [2]:
# === Cellule 2 : Winsorisation des features et log-transform de la target ===

# Sélection des colonnes numériques
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Copie du dataframe pour winsorisation
df_winsor = df.copy()

# Winsorisation aux 5% et 95%
for col in numeric_cols:
    df_winsor[col] = winsorize(df[col], limits=[0.05, 0.05])

# Log-transform de la target 'Montant'
df_winsor['Montant_log'] = np.log1p(df_winsor['Montant'])

# Vérification
print("Statistiques après winsorisation et log-transform :")
display(df_winsor[numeric_cols + ['Montant_log']].describe())


Statistiques après winsorisation et log-transform :


,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,Nombre de Titres,Montant,Echéance,Taux,Montant_log
count,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000
mean,14.850200,6.600730,26.628509,2.550155,1.954409,7597.774165,4.133039,33.962599,8.627887,1.307947
std,9.057001,3.394496,14.508804,1.103134,1.474442,10679.660190,4.635870,36.140342,0.759614,0.783107
min,1.000000,1.000000,3.000000,1.000000,0.000000,281.000000,0.225000,6.000000,7.250000,0.202941
25%,7.000000,4.000000,14.000000,2.000000,1.000000,1078.000000,1.000000,10.000000,8.030000,0.693147
50%,14.000000,7.000000,27.000000,3.000000,2.000000,3180.000000,2.001000,21.000000,8.970000,1.098946
75%,23.000000,10.000000,40.000000,4.000000,3.000000,8986.500000,5.500000,32.000000,9.010000,1.871802
max,30.000000,12.000000,49.000000,4.000000,4.000000,41689.000000,18.000000,145.000000,9.890000,2.944439


In [3]:
# === Cellule 3 : Enregistrement de la DataFrame winsorisée ===
output_path = r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale_winsor.csv"
df_winsor.to_csv(output_path, index=False)
print(f"La DataFrame winsorisée a été enregistrée dans : {output_path}")


La DataFrame winsorisée a été enregistrée dans : C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale_winsor.csv


In [4]:
# === Cellule 4 : Séparation des features et de la target + train/test split ===

# Features (toutes les colonnes sauf target originale et log-transform)
X = df_winsor.drop(columns=['Montant', 'Montant_log'])
# Target log-transformée
y = df_winsor['Montant_log']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Vérification des dimensions
print(f"Dimensions X_train : {X_train.shape}")
print(f"Dimensions X_test  : {X_test.shape}")
print(f"Dimensions y_train : {y_train.shape}")
print(f"Dimensions y_test  : {y_test.shape}")


Dimensions X_train : (16213, 12)
Dimensions X_test  : (4054, 12)
Dimensions y_train : (16213,)
Dimensions y_test  : (4054,)


In [5]:
# === Cellule 5 : Fonction pour entraîner un modèle avec RandomizedSearchCV rapide ===
def train_model(model, param_dist, X_train, y_train, X_test, y_test, n_iter=20, cv=3):
    """
    Entraîne un modèle avec RandomizedSearchCV et retourne le meilleur modèle et le RMSE.
    """
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring='neg_root_mean_squared_error',
        cv=cv,
        n_jobs=-1,
        random_state=42
    )
    # Entraînement
    random_search.fit(X_train, y_train)
    
    # Meilleur modèle
    best_model = random_search.best_estimator_
    
    # Prédiction sur X_test
    y_pred_log = best_model.predict(X_test)
    y_pred = np.expm1(y_pred_log)  # Retour à l'échelle originale
    
    # Calcul RMSE sur l'échelle originale
    rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))
    
    return best_model, rmse, random_search.best_params_
